# 2025 Indonesian Parliament Protests - Crawling


## Overview
This notebook is used to crawl X data using keyword-based searches and export the results into CSV files stored in Google Drive.

## Steps

### 1. Install Dependencies
Run the first setup cells to:
- Install Python libraries
- Install Node.js
- Mount Google Drive

### 2. Configure Parameters
Edit the configuration section:
- `TWITTER_AUTH_TOKEN` → your Twitter/X auth token
- `DATE_START` and `DATE_END` → crawling date range
- `LIMIT` → maximum tweets per query
- `KEYWORDS_A` and `KEYWORDS_B` → keywords or hashtags to crawl

### 3. Run Crawling
Execute the crawling cells to:
- Search tweets based on keywords
- Save CSV results automatically to Google Drive

### 4. Merge & Validate Data
Run the merge section to:
- Combine all CSV files
- Remove invalid/empty files
- Export the final merged dataset

## Output
The final dataset will be saved as CSV files inside your Google Drive folder.


## 1. Install & Mount Google Drive

In [ ]:
import subprocess, sys

subprocess.run([sys.executable,'-m','pip','install','pandas','-q'], check=True)

subprocess.run([
    'apt-get','install','-y','-q',
    'libnss3','libatk1.0-0','libatk-bridge2.0-0','libcups2','libdrm2',
    'libxcomposite1','libxdamage1','libxrandr2','libgbm1','libasound2',
    'libpangocairo-1.0-0','libxss1','libgtk-3-0','libx11-xcb1','libxshmfence1'
], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

subprocess.run('curl -fsSL https://deb.nodesource.com/setup_18.x | bash -',
    shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['apt-get','install','-y','-q','nodejs'],
    check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

v = subprocess.run(['node','--version'], capture_output=True, text=True).stdout.strip()

from google.colab import drive
drive.mount('/content/drive')

## 2. Config

In [ ]:
import os, time, random, shutil, pandas as pd
from datetime import datetime, timedelta
from IPython import get_ipython

TWITTER_AUTH_TOKEN = '' # change with your own token

KEYWORD_GROUP = 'ALL'

GDRIVE_FOLDER = '/content/drive/MyDrive/' # change with your own folder

DATE_START = '2025-08-15'
DATE_END   = '2025-09-09'

LIMIT     = 5000
DELAY_MIN = 22
DELAY_MAX = 35
TABS      = ['LATEST', 'TOP']

KEYWORDS_A = [
        "#BubarkanDPR",
        "#DesakPrabowoBubarkanDPR",
        "#DemoDPR",
        "#Demo",
        "#DPR",
        "#Demo25Agustus",
        "#Demo28Agustus",
        "#DPRTamak",
        "bubarkan DPR",
        "Demo DPR",
        "DPR joget",
        "tunjangan DPR",
        "Protes DPR",
        "unjuk rasa DPR",
        "aksi depan DPR"
    ]

KEYWORDS_B = [
        "#AffanKurniawan",
        "#JusticeForAffan",
        "#RantisOjol",
        "#PolisiPembunuhRakyat",
        "#PolisiPembunuh",
        "#PenjarahanDPR",
        "Affan Kurniawan",
        "mati dilindas",
        "ojol tewas",
        "penjarahan DPR",
        "rantis brimob",
        "Ahmad Sahroni DPR",
        "warga jaga warga",
        "gas air mata DPR"
    ]

if KEYWORD_GROUP == 'A':
    KEYWORDS = KEYWORDS_A
elif KEYWORD_GROUP == 'B':
    KEYWORDS = KEYWORDS_B
else:
    KEYWORDS = KEYWORDS_A + KEYWORDS_B
os.makedirs(GDRIVE_FOLDER, exist_ok=True)
os.makedirs('/content/tweets-data', exist_ok=True)

def date_pairs(s, e):
    start, end, pairs = datetime.strptime(s,'%Y-%m-%d'), datetime.strptime(e,'%Y-%m-%d'), []
    cur = start
    while cur < end:
        pairs.append((cur.strftime('%Y-%m-%d'), (cur+timedelta(days=1)).strftime('%Y-%m-%d')))
        cur += timedelta(days=1)
    return pairs

DATE_PAIRS = date_pairs(DATE_START, DATE_END)
total_tasks = len(KEYWORDS) * len(TABS) * len(DATE_PAIRS)
est = total_tasks * (DELAY_MIN+DELAY_MAX)/2/60

print(f'Group        : {KEYWORD_GROUP}  ({len(KEYWORDS)} keyword)')
print(f'Range     : {DATE_START} → {DATE_END}  ({len(DATE_PAIRS)} hari)')
print(f'Tab         : {TABS}')
print(f'Total task  : {total_tasks}')
print(f'Estimate    : ~{est:.0f} mnt (~{est/60:.1f} jam)')
print(f'Drive folder: {GDRIVE_FOLDER}')
print()
for i,kw in enumerate(KEYWORDS,1): print(f'  {i:>2}. {kw}')

## 3. Data Crawling


In [ ]:
def safe_fn(t): return t.replace('#','').replace(' ','_').replace('/','_').replace('+','plus')[:35]
def drv_path(kw,tab,since): return f'{GDRIVE_FOLDER}/{safe_fn(kw)}_{tab}_{since}.csv'
def drv_exists(kw,tab,since): return os.path.exists(drv_path(kw,tab,since))

def save_drive(local_file, kw, tab, since):
    dest = drv_path(kw, tab, since)
    if local_file and os.path.exists(local_file):
        shutil.copy2(local_file, dest)
        return True
    open(dest,'w').close()
    return False

def crawl_one(kw, tab, since, until, out_name):
    query = f'{kw} since:{since} until:{until} lang:en'
    cmd = (f'npx -y tweet-harvest@2.6.1 -o "{out_name}" '
           f'-s "{query}" --tab "{tab}" -l {LIMIT} --token {TWITTER_AUTH_TOKEN}')
    get_ipython().system(cmd)
    fp = f'/content/tweets-data/{out_name}.csv'
    if os.path.exists(fp):
        try: return len(pd.read_csv(fp, on_bad_lines='skip')), fp
        except: return 0, fp
    return 0, None

print('='*55)
print(f' CRAWLING — Group {KEYWORD_GROUP}')
print('='*55)

done=skipped=failed=total_tw=0
start_t = time.time()
total = len(KEYWORDS)*len(TABS)*len(DATE_PAIRS)

for since, until in DATE_PAIRS:
    for tab in TABS:
        for kw in KEYWORDS:
            done += 1
            out_name = f'{safe_fn(kw)}_{tab}_{since}'

            if drv_exists(kw, tab, since):
                skipped += 1
                print(f'  [SKIP {done}/{total}] {tab} | {since} | {kw}')
                continue

            ela = (time.time()-start_t)/60
            eta = (total-done)*(DELAY_MIN+DELAY_MAX)/2/60
            print(f'\n[{done}/{total}] {tab} | {since} | {kw}')
            print(f'  Query: {kw} since:{since} until:{until} lang:en')
            print(f'  Time: ETA ~{eta:.0f} minutes')

            try:
                n, lf = crawl_one(kw, tab, since, until, out_name)
                saved = save_drive(lf, kw, tab, since)
                total_tw += n
                print(f'  {"✅" if n>0 else "⚠️ "} {n} tweet | Drive: {"OK" if saved else "FAILED"} | Total: {total_tw:,}')
                if lf and os.path.exists(lf): os.remove(lf)
            except Exception as e:
                failed += 1
                save_drive(None, kw, tab, since)
                print(f'Error: {e}')

            d = random.randint(DELAY_MIN, DELAY_MAX)
            print(f'Pause {d}s: ', end='')
            for i in range(d):
                time.sleep(1)
                if (i+1)%10==0: print(f'{i+1}', end=' ', flush=True)
            print('OK')

print(f'\n{"="*55}')
print(f' FINISH — Group {KEYWORD_GROUP}')
print(f'{"="*55}')
print(f'  Total task  : {total}')
print(f'  Skipped     : {skipped}')
print(f'  Failed       : {failed}')
print(f'  Total tweet : {total_tw:,}')
print(f'  Time      : {(time.time()-start_t)/60:.1f} minutes')


## 4. Merge, Validate, and Download

In [ ]:
import pandas as pd, os, re
from datetime import datetime
from google.colab import files

START_DATE = datetime.strptime(DATE_START, "%Y-%m-%d").date()
END_DATE   = datetime.strptime(DATE_END, "%Y-%m-%d").date()

csvs = []

for f in os.listdir(GDRIVE_FOLDER):

    if not f.endswith(".csv"):
        continue
    if "merged" in f:
        continue

    m = re.search(r'(\d{4}-\d{2}-\d{2})', f)
    if not m:
        continue

    file_date = datetime.strptime(m.group(1), "%Y-%m-%d").date()

    if START_DATE <= file_date < END_DATE:
        csvs.append(os.path.join(GDRIVE_FOLDER, f))

csvs = sorted(csvs)

dfs, empty = [], 0

for fp in csvs:

    if os.path.getsize(fp) == 0:
        empty += 1
        continue

    try:
        df = pd.read_csv(
            fp,
            on_bad_lines='skip',
            dtype={
                'conversation_id_str': str,
                'id_str': str,
                'user_id_str': str
            }
        )

        if len(df) > 0:
            dfs.append(df)

    except Exception as e:
        print(f"⚠ {os.path.basename(fp)}: {e}")

print(f"Empty file: {empty}")
print(f"Data file: {len(dfs)}")

if not dfs:
    print("There is no data to merge.")
else:

    df_all = pd.concat(dfs, ignore_index=True)

    before = len(df_all)

    df_all = df_all.drop_duplicates(subset=['id_str'])

    after = len(df_all)

    print(f"Before deduplication : {before:,}")
    print(f"Aftter deduplication : {after:,}")
    print(f"Duplication          : {before-after:,}")

    if 'created_at' in df_all.columns:

        df_all['_d'] = pd.to_datetime(
            df_all['created_at'],
            format='%a %b %d %H:%M:%S %z %Y',
            errors='coerce'
        ).dt.date

        print("\nDate distribution:")
        print(df_all['_d'].value_counts().sort_index())

        df_all = df_all.drop(columns=['_d'])

    out = "/content/merge.csv"

    df_all.to_csv(out, index=False)
    df_all.to_csv(f"{GDRIVE_FOLDER}/merge_MASTER.csv", index=False)

    print(out)
    print(f"{GDRIVE_FOLDER}/merge_MASTER.csv")

    files.download(out)